In [1]:
# Imports and configuration

from pathlib import Path
import os
import hashlib

import pandas as pd
import mysql.connector
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

CLEAN_FILE = PROJECT_ROOT / "Cleaned_Data" / "nfhs_cleaned.csv"

SQL_DIR = PROJECT_ROOT / "SQL"
REPORT_DIR = (
    PROJECT_ROOT
    / "Reports"
    / "04_mysql_load_and_validation"
)

SQL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COLUMN_MAPPING_FILE = REPORT_DIR / "column_mapping.csv"
CREATE_TABLE_FILE = SQL_DIR / "01_Create_table.sql"

DB_NAME = "nfhs5_healthcare"
TABLE_NAME = "nfhs_district_data"
BATCH_SIZE = 500

load_dotenv(PROJECT_ROOT / ".env")

DB_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "port": int(os.getenv("MYSQL_PORT", "3306")),
    "user": os.getenv("MYSQL_USER"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE", DB_NAME),
}

if not CLEAN_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {CLEAN_FILE}. Run Notebook 3 first."
    )

if not DB_CONFIG["user"] or DB_CONFIG["password"] is None:
    raise ValueError("Check MYSQL_USER and MYSQL_PASSWORD in your .env file.")

print("Configuration loaded.")
print(f"Cleaned dataset: {CLEAN_FILE}")
print(f"SQL scripts: {SQL_DIR}")
print(f"Reports: {REPORT_DIR}")

Configuration loaded.
Cleaned dataset: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Cleaned_Data\nfhs_cleaned.csv
SQL scripts: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\SQL
Reports: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\04_mysql_load_and_validation


In [2]:
# Load and validate the cleaned CSV

if not CLEAN_FILE.exists():
    raise FileNotFoundError(f"Cleaned dataset not found: {CLEAN_FILE}")

df = pd.read_csv(
    CLEAN_FILE,
    dtype={"district_names": "string", "state_ut": "string"}
)

required_columns = {"district_names", "state_ut"}

if not required_columns.issubset(df.columns):
    raise ValueError(f"Required columns missing: {required_columns - set(df.columns)}")

if df[["district_names", "state_ut"]].isna().any().any():
    raise ValueError("Missing district or state identifiers found.")

if df[["district_names", "state_ut"]].apply(
    lambda col: col.str.strip().eq("").any()
).any():
    raise ValueError("Blank district or state identifiers found.")

if df.duplicated(["district_names", "state_ut"]).any():
    raise ValueError("Duplicate district-state pairs found.")

if df["state_ut"].eq("Maharastra").any():
    raise ValueError("Old state spelling 'Maharastra' remains in cleaned CSV.")

assert len(df) == 707, f"Expected 707 rows; found {len(df)}."

print("Cleaned dataset loaded and validated.")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Maharashtra rows:", int(df["state_ut"].eq("Maharashtra").sum()))

display(df.head())

Cleaned dataset loaded and validated.
Rows: 707
Columns: 109
Maharashtra rows: 36


,district_names,state_ut,number_of_households_surveyed,number_of_women_age_15_49_years_interviewed,number_of_men_age_15_54_years_interviewed,female_population_age_6_years_and_above_who_ever_attended_school,population_below_age_15_years,sex_ratio_of_the_total_population_females_per_1_000_males,sex_ratio_at_birth_for_children_born_in_the_last_five_years_females_per_1_000_males,children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority,...,men_age_15_years_and_above_wih_mildly_elevated_blood_pressure_systolic_140_159_mm_of_hg_and_or_diastolic_90_99_mm_of_hg,men_age_15_years_and_above_wih_moderately_or_severely_elevated_blood_pressure_systolic_160_mm_of_hg_and_or_diastolic_100_mm_of_hg,men_age_15_years_and_above_wih_elevated_blood_pressure_systolic_140_mm_of_hg_and_or_diastolic_90_mm_of_hg_or_taking_medicine_to_control_blood_pressure,women_age_30_49_years_ever_undergone_a_screening_test_for_cervical_cancer,women_age_30_49_years_ever_undergone_a_breast_examination_for_breast_cancer,women_age_30_49_years_ever_undergone_an_oral_cavity_examination_for_oral_cancer,women_age_15_years_and_above_who_use_any_kind_of_tobacco,men_age_15_years_and_above_who_use_any_kind_of_tobacco,women_age_15_years_and_above_who_consume_alcohol,men_age_15_years_and_above_who_consume_alcohol
0,Nicobars,Andaman & Nicobar Islands,882,764,125,78.01,22.98,973.31,927.41,98.01,...,32.86,11.06,46.97,13.35,13.16,5.37,63.46,76.79,29.59,64.49
1,North & Middle Andaman,Andaman & Nicobar Islands,874,789,108,82.66,19.82,949.82,844.43,100.00,...,22.62,5.97,32.20,1.70,0.25,15.84,46.77,70.47,5.08,45.26
2,South Andaman,Andaman & Nicobar Islands,868,844,134,84.68,20.95,967.48,934.92,96.53,...,17.88,6.09,26.90,1.32,0.67,8.00,19.60,50.76,1.72,32.81
3,Srikakulam,Andhra Pradesh,874,780,100,59.99,20.71,1139.51,1162.58,94.96,...,14.37,5.51,22.89,1.04,0.23,3.76,7.09,21.32,0.55,28.26
4,Vizianagaram,Andhra Pradesh,902,853,134,55.95,20.56,1114.35,898.03,95.42,...,14.83,6.43,25.13,4.90,0.63,7.33,11.41,21.47,0.80,32.30


In [3]:
# Create column mapping and SQL schema

def mysql_column_name(name):
    safe_name = "".join(
        char.lower() if char.isalnum() else "_"
        for char in str(name)
    )
    safe_name = "_".join(part for part in safe_name.split("_") if part)
    return safe_name or "column"


column_mapping = []
used_names = set()

for original_name in df.columns:
    sql_name = mysql_column_name(original_name)

    # Ensure unique names and avoid excessively long identifiers.
    if len(sql_name) > 55:
        suffix = hashlib.md5(
            str(original_name).encode("utf-8")
        ).hexdigest()[:8]
        sql_name = f"{sql_name[:46]}_{suffix}"

    base_name = sql_name
    counter = 1

    while sql_name in used_names:
        sql_name = f"{base_name[:55]}_{counter}"
        counter += 1

    used_names.add(sql_name)

    column_mapping.append({
        "original_column": original_name,
        "mysql_column": sql_name
    })

mapping_df = pd.DataFrame(column_mapping)
mapping_df.to_csv(COLUMN_MAPPING_FILE, index=False)

sql_columns = []

for item in column_mapping:
    original_name = item["original_column"]
    sql_name = item["mysql_column"]

    quoted_name = f"`{sql_name}`"

    if original_name in ["district_names", "state_ut"]:
        definition = f"{quoted_name} VARCHAR(150) NOT NULL"
    else:
        definition = f"{quoted_name} DECIMAL(18,3) NULL"

    sql_columns.append(definition)

create_table_sql = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_NAME}` (
    {", ".join(sql_columns)},
    UNIQUE KEY `uq_district_state` (`{mysql_column_name("district_names")}`, `{mysql_column_name("state_ut")}`)
);
"""

CREATE_TABLE_FILE.write_text(create_table_sql, encoding="utf-8")

print("Column mapping saved:", COLUMN_MAPPING_FILE)
print("Table schema saved:", CREATE_TABLE_FILE)

display(mapping_df.head())

Column mapping saved: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\04_mysql_load_and_validation\column_mapping.csv
Table schema saved: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\SQL\01_Create_table.sql


,original_column,mysql_column
0,district_names,district_names
1,state_ut,state_ut
2,number_of_households_surveyed,number_of_households_surveyed
3,number_of_women_age_15_49_years_interviewed,number_of_women_age_15_49_years_interviewed
4,number_of_men_age_15_54_years_interviewed,number_of_men_age_15_54_years_interviewed


In [4]:
# Connect to MySQL

connection = mysql.connector.connect(**DB_CONFIG)
cursor = connection.cursor()

print(f"Connected to database: {DB_CONFIG['database']}")

Connected to database: nfhs5_healthcare


In [5]:
# Create the table and check its schema

cursor.execute(create_table_sql)
connection.commit()

cursor.execute(f"SHOW COLUMNS FROM `{TABLE_NAME}`")
existing_columns = [row[0] for row in cursor.fetchall()]

expected_columns = mapping_df["mysql_column"].tolist()

if existing_columns != expected_columns:
    raise ValueError(
        "The existing MySQL table schema differs from the expected schema. "
        "Review SHOW COLUMNS and the generated SQL before loading data. "
        "Do not drop the table until you have confirmed that replacing it is safe."
    )

print(f"Table `{TABLE_NAME}` exists with the expected column order.")

Table `nfhs_district_data` exists with the expected column order.


In [6]:
# Load data in batches

mysql_df = df.copy()

# Convert pandas missing values to Python None for MySQL.
for column in mysql_df.columns:
    if column not in ["district_names", "state_ut"]:
        mysql_df[column] = pd.to_numeric(mysql_df[column], errors="coerce")

mysql_df = mysql_df.astype(object).where(pd.notna(mysql_df), None)

mysql_df.columns = mapping_df["mysql_column"].tolist()

column_names = list(mysql_df.columns)
quoted_columns = ", ".join(f"`{column}`" for column in column_names)
placeholders = ", ".join(["%s"] * len(column_names))

insert_sql = (
    f"INSERT INTO `{TABLE_NAME}` ({quoted_columns}) "
    f"VALUES ({placeholders})"
)

cursor.execute(f"TRUNCATE TABLE `{TABLE_NAME}`")

inserted_rows = 0

for start in range(0, len(mysql_df), BATCH_SIZE):
    batch = mysql_df.iloc[start:start + BATCH_SIZE]
    rows = [
        tuple(None if pd.isna(value) else value for value in row)
        for row in batch.itertuples(index=False, name=None)
    ]

    cursor.executemany(insert_sql, rows)
    connection.commit()
    inserted_rows += len(rows)

print("Rows inserted:", inserted_rows)

Rows inserted: 707


In [7]:
# Validate the loaded MySQL table

cursor.execute(f"SELECT COUNT(*) FROM `{TABLE_NAME}`")
database_row_count = cursor.fetchone()[0]

district_col = mysql_column_name("district_names")
state_col = mysql_column_name("state_ut")

cursor.execute(
    f"""
    SELECT COUNT(*)
    FROM `{TABLE_NAME}`
    WHERE `{district_col}` IS NULL OR TRIM(`{district_col}`) = ''
       OR `{state_col}` IS NULL OR TRIM(`{state_col}`) = ''
    """
)
missing_identifier_count = cursor.fetchone()[0]

cursor.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT `{district_col}`, `{state_col}`
        FROM `{TABLE_NAME}`
        GROUP BY `{district_col}`, `{state_col}`
        HAVING COUNT(*) > 1
    ) AS duplicates
    """
)
duplicate_pair_count = cursor.fetchone()[0]

cursor.execute(
    f"""
    SELECT COUNT(*)
    FROM `{TABLE_NAME}`
    WHERE `{state_col}` = 'Maharastra'
    """
)
old_spelling_count = cursor.fetchone()[0]

cursor.execute(
    f"""
    SELECT COUNT(*)
    FROM `{TABLE_NAME}`
    WHERE `{state_col}` = 'Maharashtra'
    """
)
maharashtra_count = cursor.fetchone()[0]

print("Database rows:", database_row_count)
print("Missing identifiers:", missing_identifier_count)
print("Duplicate district-state pairs:", duplicate_pair_count)
print("Maharashtra rows:", maharashtra_count)
print("Maharastra rows remaining:", old_spelling_count)

assert database_row_count == 707
assert missing_identifier_count == 0
assert duplicate_pair_count == 0
assert old_spelling_count == 0
assert maharashtra_count == 36

print("MySQL validation passed.")

Database rows: 707
Missing identifiers: 0
Duplicate district-state pairs: 0
Maharashtra rows: 36
Maharastra rows remaining: 0
MySQL validation passed.


In [8]:
# Preview data and close the connection

cursor.execute(
    f"""
    SELECT `{district_col}`, `{state_col}`
    FROM `{TABLE_NAME}`
    ORDER BY `{state_col}`, `{district_col}`
    LIMIT 10
    """
)

preview = pd.DataFrame(
    cursor.fetchall(),
    columns=["district_names", "state_ut"]
)

display(preview)

cursor.close()
connection.close()

print("MySQL connection closed.")
print(f"Notebook 4 reports: {REPORT_DIR}")

,district_names,state_ut
0,Nicobars,Andaman & Nicobar Islands
1,North & Middle Andaman,Andaman & Nicobar Islands
2,South Andaman,Andaman & Nicobar Islands
3,Anantapur,Andhra Pradesh
4,Chittoor,Andhra Pradesh
5,East Godavari,Andhra Pradesh
6,Guntur,Andhra Pradesh
7,Krishna,Andhra Pradesh
8,Kurnool,Andhra Pradesh
9,Prakasam,Andhra Pradesh


MySQL connection closed.
Notebook 4 reports: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\04_mysql_load_and_validation
